# Vivado Area & Power Analysis — Addition Comparison

This notebook compares Vivado results for the Conventional Binary, LFSR stochastic, and Sobol stochastic adders using the screenshots stored in this folder.

The notebook discovers the screenshots, displays them, extracts common resource/power values using OCR when available, allows manual correction, and generates multiple separate plots for area and power.

Important: OCR is only a convenience. Values used in a formal report should be verified against the original Vivado screenshots.

## Metrics

Area/resource metrics: Slice LUTs, LUT as Logic, Slice Registers, BRAM, DSP, I/O and BUFG when reported.

Power metrics: Total On-Chip Power, Dynamic Power, Static Power, Clock Power, Logic Power, Signal Power, BRAM Power, DSP Power and I/O Power when reported.

Power is normalized to mW.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path('.')
ARCH_DIRS = {
    'Binary': ROOT / 'binary adder',
    'LFSR': ROOT / 'lfsr strochastic',
    'Sobol': ROOT / 'sobol adder'
}

print('Working directory:', ROOT.resolve())
for arch, folder in ARCH_DIRS.items():
    print(f'{arch}: {folder} | exists={folder.exists()}')

In [ ]:
def infer_width(name):
    s = name.lower()
    m = re.search(r'(8|16|32)\s*bit', s)
    if m:
        return int(m.group(1))
    m = re.search(r'(^|\D)(8|16|32)(?=\s*(?:area|power))', s)
    return int(m.group(2)) if m else np.nan

def infer_report(name):
    s = name.lower()
    if 'power' in s:
        return 'power'
    if 'area' in s:
        return 'area'
    return 'unknown'

inventory = []
for arch, folder in ARCH_DIRS.items():
    if folder.exists():
        for p in sorted(folder.glob('*.png')):
            inventory.append({
                'architecture': arch,
                'width': infer_width(p.name),
                'report': infer_report(p.name),
                'file': str(p),
                'name': p.name
            })

inventory_df = pd.DataFrame(inventory).sort_values(['report','architecture','width','name'])
display(inventory_df)

In [ ]:
for row in inventory_df.itertuples(index=False):
    print(f'{row.architecture:8s} | {str(row.width):>4s} bit | {row.report:7s} | {row.name}')
    display(Image(filename=row.file, width=850))

## OCR extraction

Install OCR support locally with: pip install pytesseract pillow

Linux also needs the Tesseract executable. If OCR is unavailable, use the manual override cell later in this notebook.

In [ ]:
try:
    import pytesseract
    from PIL import Image as PILImage, ImageOps, ImageEnhance, ImageFilter
    OCR_AVAILABLE = True
except Exception as exc:
    OCR_AVAILABLE = False
    print('OCR unavailable:', exc)

def preprocess(path):
    img = PILImage.open(path).convert('L')
    img = ImageOps.autocontrast(img)
    img = img.resize((img.width * 2, img.height * 2))
    img = ImageEnhance.Contrast(img).enhance(1.4)
    img = ImageEnhance.Sharpness(img).enhance(1.8)
    return img.filter(ImageFilter.SHARPEN)

ocr_text = {}
if OCR_AVAILABLE:
    for row in inventory_df.itertuples(index=False):
        txt = pytesseract.image_to_string(preprocess(row.file), config='--psm 6')
        ocr_text[row.file] = txt
        print('\n' + '=' * 90)
        print(row.name)
        print(txt[:5000])
else:
    print('Run pip install pytesseract pillow and install Tesseract to enable automatic extraction.')

In [ ]:
def power_mw(text, labels):
    for label in labels:
        pattern = rf'{label}[^0-9]{{0,100}}([0-9]+(?:\.[0-9]+)?)\s*(uW|mW|W)'
        m = re.search(pattern, text, flags=re.I)
        if m:
            value = float(m.group(1))
            unit = m.group(2).lower()
            if unit == 'w':
                return value * 1000.0
            if unit == 'uw':
                return value / 1000.0
            return value
    return np.nan

def integer_metric(text, labels):
    for label in labels:
        pattern = rf'{label}[^0-9]{{0,100}}([0-9]+)'
        m = re.search(pattern, text, flags=re.I)
        if m:
            return float(m.group(1).replace(',', ''))
    return np.nan

def parse_metrics(text, report):
    out = {}
    if report == 'power':
        out['total_power_mw'] = power_mw(text, ['total\\s+(?:on[- ]chip\\s+)?power'])
        out['dynamic_power_mw'] = power_mw(text, ['total\\s+dynamic\\s+power'])
        out['static_power_mw'] = power_mw(text, ['total\\s+static\\s+power'])
        out['clock_power_mw'] = power_mw(text, ['clock\\s+power'])
        out['logic_power_mw'] = power_mw(text, ['logic\\s+power'])
        out['signal_power_mw'] = power_mw(text, ['signal\\s+power'])
        out['bram_power_mw'] = power_mw(text, ['(?:block\\s*ram|bram)\\s+power'])
        out['dsp_power_mw'] = power_mw(text, ['dsp\\s+power'])
        out['io_power_mw'] = power_mw(text, ['i/?o\\s+power'])
    elif report == 'area':
        out['slice_luts'] = integer_metric(text, ['slice\\s+luts?', 'luts?'])
        out['lut_as_logic'] = integer_metric(text, ['lut\\s+as\\s+logic'])
        out['slice_registers'] = integer_metric(text, ['slice\\s+registers?', 'flip[- ]flops?', '\\bffs?\\b'])
        out['block_ram'] = integer_metric(text, ['block\\s+ram\\s+tile', 'bram'])
        out['dsp'] = integer_metric(text, ['\\bdsp(?:s|48)?\\b'])
        out['iobuf'] = integer_metric(text, ['i/?o\\s+buf', '\\bio\\b'])
        out['bufg'] = integer_metric(text, ['\\bbufg\\b'])
    return out

records = []
for row in inventory_df.itertuples(index=False):
    records.append({
        'architecture': row.architecture,
        'width': row.width,
        'report': row.report,
        'file': row.file,
        **parse_metrics(ocr_text.get(row.file, ''), row.report)
    })

raw_df = pd.DataFrame(records)
power_df = raw_df[raw_df['report'] == 'power'].copy()
area_df = raw_df[raw_df['report'] == 'area'].copy()

display(power_df)
display(area_df)

## Manual verification / correction

OCR can confuse decimal points and table columns. Enter any values that need correction below. A supplied value replaces the OCR value for that architecture and width.

In [ ]:
manual_power = pd.DataFrame([
    # Example:
    # {'architecture':'Binary', 'width':8, 'total_power_mw':1.234},
])

manual_area = pd.DataFrame([
    # Example:
    # {'architecture':'Binary', 'width':8, 'slice_luts':10},
])

def apply_overrides(base, overrides):
    result = base.copy()
    if overrides.empty:
        return result
    for _, r in overrides.iterrows():
        mask = (result['architecture'] == r['architecture']) & (result['width'] == r['width'])
        for col in overrides.columns:
            if col not in ['architecture','width'] and pd.notna(r[col]):
                result.loc[mask, col] = r[col]
    return result

power_df = apply_overrides(power_df, manual_power)
area_df = apply_overrides(area_df, manual_area)
display(power_df)
display(area_df)

# Area/resource plots

In [ ]:
def plot_vs_width(df, metric, title, ylabel):
    fig, ax = plt.subplots(figsize=(8,5))
    for arch, g in df.dropna(subset=[metric]).groupby('architecture'):
        g = g.sort_values('width')
        ax.plot(g['width'], g[metric], marker='o', label=arch)
    ax.set_xlabel('Input width (bits)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

for metric, title, ylabel in [
    ('slice_luts', 'Slice LUT Usage vs Input Width', 'Slice LUTs'),
    ('lut_as_logic', 'LUT-as-Logic Usage vs Input Width', 'LUTs'),
    ('slice_registers', 'Slice Register Usage vs Input Width', 'Registers'),
    ('block_ram', 'Block RAM Usage vs Input Width', 'BRAM tiles'),
    ('dsp', 'DSP Usage vs Input Width', 'DSPs')
]:
    if metric in area_df.columns:
        plot_vs_width(area_df, metric, title, ylabel)

# Power plots

In [ ]:
for metric, title in [
    ('total_power_mw', 'Total On-Chip Power vs Input Width'),
    ('dynamic_power_mw', 'Dynamic Power vs Input Width'),
    ('static_power_mw', 'Static Power vs Input Width'),
    ('clock_power_mw', 'Clock Power vs Input Width'),
    ('logic_power_mw', 'Logic Power vs Input Width'),
    ('signal_power_mw', 'Signal Power vs Input Width'),
    ('bram_power_mw', 'BRAM Power vs Input Width'),
    ('dsp_power_mw', 'DSP Power vs Input Width'),
    ('io_power_mw', 'I/O Power vs Input Width')
]:
    if metric in power_df.columns:
        plot_vs_width(power_df, metric, title, 'Power (mW)')

## Power-component analysis

Component plots help explain a total-power difference. For the stochastic adders, relate the measured components to the extra sequence-generator, comparator, MUX and switching hardware. Do not attribute a measured power value to behavioral RTL style alone; Vivado reports the synthesized/implemented design and its activity assumptions.

In [ ]:
components = ['clock_power_mw','logic_power_mw','signal_power_mw','bram_power_mw','dsp_power_mw','io_power_mw']

for arch in sorted(power_df['architecture'].dropna().unique()):
    for width in sorted(power_df.loc[power_df['architecture']==arch, 'width'].dropna().unique()):
        r = power_df[(power_df['architecture']==arch) & (power_df['width']==width)]
        if r.empty:
            continue
        s = r.iloc[0][components].dropna()
        if s.empty:
            continue
        fig, ax = plt.subplots(figsize=(8,5))
        ax.bar(s.index.str.replace('_mw','', regex=False), s.values)
        ax.set_title(f'{arch} — Power Components — {int(width)} bit')
        ax.set_ylabel('Power (mW)')
        ax.tick_params(axis='x', rotation=35)
        plt.tight_layout()
        plt.show()

## Normalized total power

For each width, binary is the reference only for normalization.

Normalized power = design power / binary power.

A value of 1.0 corresponds to the binary reference at that width.

In [ ]:
pivot = power_df.pivot_table(index='width', columns='architecture', values='total_power_mw', aggfunc='first')
for arch in ['LFSR','Sobol']:
    if arch in pivot.columns and 'Binary' in pivot.columns:
        pivot[arch + '_normalized'] = pivot[arch] / pivot['Binary']

display(pivot)

fig, ax = plt.subplots(figsize=(8,5))
for arch in ['LFSR','Sobol']:
    col = arch + '_normalized'
    if col in pivot.columns:
        ax.plot(pivot.index, pivot[col], marker='o', label=arch)
ax.axhline(1.0, linestyle='--')
ax.set_xlabel('Input width (bits)')
ax.set_ylabel('Normalized total power')
ax.set_title('Stochastic Adder Power Normalized to Binary')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Area versus power

This plot visualizes the hardware trade-off by placing total on-chip power against the primary area metric, Slice LUTs, when both measurements are available.

In [ ]:
trade = power_df.merge(area_df, on=['architecture','width'], how='inner', suffixes=('_power','_area'))

fig, ax = plt.subplots(figsize=(8,6))
for arch, g in trade.groupby('architecture'):
    g = g.dropna(subset=['total_power_mw','slice_luts'])
    ax.scatter(g['slice_luts'], g['total_power_mw'], label=arch)
    for _, r in g.iterrows():
        ax.annotate(f'{int(r.width)}b', (r.slice_luts, r.total_power_mw), xytext=(5,5), textcoords='offset points')
ax.set_xlabel('Slice LUTs')
ax.set_ylabel('Total On-Chip Power (mW)')
ax.set_title('Area vs Total Power')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Percentage-change table

This table gives the measured change of LFSR and Sobol total power relative to the binary result at the same width.

Percentage change = (design - binary) / binary × 100.

In [ ]:
p = power_df.pivot_table(index='width', columns='architecture', values='total_power_mw', aggfunc='first').reset_index()
rows=[]
for _, r in p.iterrows():
    base = r.get('Binary', np.nan)
    if pd.isna(base) or base == 0:
        continue
    for arch in ['LFSR','Sobol']:
        v = r.get(arch, np.nan)
        if pd.notna(v):
            rows.append({
                'width': int(r['width']),
                'architecture': arch,
                'binary_power_mw': base,
                'design_power_mw': v,
                'change_mw': v - base,
                'change_percent': (v - base) / base * 100.0
            })
comparison_df = pd.DataFrame(rows)
display(comparison_df.sort_values(['width','architecture']))

## Interpretation notes

Use the following structure in the final report:

1. Compare area scaling from 8 to 16 to 32 bits.
2. Compare total, dynamic and static power at each width.
3. Inspect clock, logic and signal components before explaining total power.
4. Relate differences to the hardware blocks present in each architecture.
5. Keep stream length L separate from input width N: in the current project, L16/L32 is primarily a testbench sampling choice, while N is the synthesized datapath/generator width.
6. Confirm the same FPGA device and timing/reporting assumptions before drawing conclusions.

Because the measurements come from Vivado screenshots, the final discussion should describe what the reported numbers show rather than assuming that behavioral or structural coding alone caused a particular power result.